## Ontology Builder

In [8]:
!pip install rdflib

In [ ]:
from rdflib import Graph, Literal, Namespace, RDF, RDFS
from rdflib.namespace import OWL, XSD
import re

BMO = Namespace("http://www.bridge-management-ontology.org/bmo#")
g = Graph()
g.bind("bmo", BMO)
g.bind("owl", OWL)
g.bind("rdfs", RDFS)
g.bind("xsd", XSD)

top_level_classes = {
    "Asset": "A physical entity that is owned and has value, such as a bridge or its components.",
    "Event": "An occurrence that happens at a specific point in time related to an asset.",
    "ExternalFactor": "An external condition or influence that affects an asset."
}
for name, comment in top_level_classes.items():
    class_uri = BMO[name]
    g.add((class_uri, RDF.type, OWL.Class))
    g.add((class_uri, RDFS.label, Literal(name)))
    g.add((class_uri, RDFS.comment, Literal(comment)))


bridge_class = BMO.Bridge
g.add((bridge_class, RDF.type, OWL.Class))
g.add((bridge_class, RDFS.subClassOf, BMO.Asset))
g.add((bridge_class, RDFS.label, Literal("Bridge")))

bridge_element_class = BMO.BridgeElement
g.add((bridge_element_class, RDF.type, OWL.Class))
g.add((bridge_element_class, RDFS.subClassOf, BMO.Asset))
g.add((bridge_element_class, RDFS.label, Literal("Bridge Element")))

inspection_event_class = BMO.InspectionEvent
g.add((inspection_event_class, RDF.type, OWL.Class))
g.add((inspection_event_class, RDFS.subClassOf, BMO.Event))
g.add((inspection_event_class, RDFS.label, Literal("Inspection Event")))

maintenance_event_class = BMO.MaintenanceEvent
g.add((maintenance_event_class, RDF.type, OWL.Class))
g.add((maintenance_event_class, RDFS.subClassOf, BMO.Event))
g.add((maintenance_event_class, RDFS.label, Literal("Maintenance Event")))

traffic_profile_class = BMO.TrafficProfile
g.add((traffic_profile_class, RDF.type, OWL.Class))
g.add((traffic_profile_class, RDFS.subClassOf, BMO.ExternalFactor))
g.add((traffic_profile_class, RDFS.label, Literal("Traffic Profile")))

environmental_exposure_class = BMO.EnvironmentalExposure
g.add((environmental_exposure_class, RDF.type, OWL.Class))
g.add((environmental_exposure_class, RDFS.subClassOf, BMO.ExternalFactor))
g.add((environmental_exposure_class, RDFS.label, Literal("Environmental Exposure")))


g.add((BMO.hasElement, RDF.type, OWL.ObjectProperty))
g.add((BMO.hasElement, RDFS.domain, bridge_class))
g.add((BMO.hasElement, RDFS.range, bridge_element_class))

g.add((BMO.undergoesEvent, RDF.type, OWL.ObjectProperty))
g.add((BMO.undergoesEvent, RDFS.domain, BMO.Asset))
g.add((BMO.undergoesEvent, RDFS.range, BMO.Event))

g.add((BMO.isInfluencedBy, RDF.type, OWL.ObjectProperty))
g.add((BMO.isInfluencedBy, RDFS.domain, BMO.Asset))
g.add((BMO.isInfluencedBy, RDFS.range, BMO.ExternalFactor))



def create_data_property(column_name, domain_class, data_type=XSD.string):
    """Helper function to create a data property and link it to its domain class."""
    prop_name_camel = ''.join(x.title() for x in re.split(r'[\s\(\).\-/]', column_name) if x)
    prop_name_camel = prop_name_camel[0].lower() + prop_name_camel[1:] if prop_name_camel else column_name.lower()
    
    prop_uri = BMO[prop_name_camel]
    g.add((prop_uri, RDF.type, OWL.DatatypeProperty))
    g.add((prop_uri, RDFS.label, Literal(column_name)))
    g.add((prop_uri, RDFS.domain, domain_class))
    g.add((prop_uri, RDFS.range, data_type))

# Asset -> Bridge Properties
create_data_property("STRUCTURE_NUMBER", bridge_class, XSD.string)
create_data_property("NBI_43A - Main Span Material", bridge_class, XSD.string)
create_data_property("NBI_107 - Deck Structure Type Code", bridge_class, XSD.string)
create_data_property("NBI_49 - Structure Length (ft.)", bridge_class, XSD.decimal)
create_data_property("NBI_CAT29 - Deck Area (sq. ft.)", bridge_class, XSD.decimal)
create_data_property("NBI_27 - Year Built", bridge_class, XSD.gYear)

# Asset -> BridgeElement Properties
create_data_property("ELEMENT_NAME", bridge_element_class, XSD.string)
create_data_property("ELEMENT_CATEGORY", bridge_element_class, XSD.string)
create_data_property("TOTAL_QUANTITY", bridge_element_class, XSD.decimal)
create_data_property("CONDITION_STATE_1", bridge_element_class, XSD.integer)
create_data_property("CONDITION_STATE_2", bridge_element_class, XSD.integer)
create_data_property("CONDITION_STATE_3", bridge_element_class, XSD.integer)
create_data_property("CONDITION_STATE_4", bridge_element_class, XSD.integer)

# Event -> InspectionEvent Properties
create_data_property("NBI_90 - Inspection Date", inspection_event_class, XSD.date)
create_data_property("NBI_91 - Designated Inspection Frequency", inspection_event_class, XSD.integer)
create_data_property("NBI_CAT10 - Bridge Condition", inspection_event_class, XSD.string)

# Event -> MaintenanceEvent Properties 
create_data_property("NBI_94 - Bridge Improvement Cost", maintenance_event_class, XSD.decimal)

# ExternalFactor -> TrafficProfile Properties
create_data_property("NBI_29 - Average Daily Traffic", traffic_profile_class, XSD.integer)
create_data_property("NBI_109 - Average Daily Truck Traffic (%)", traffic_profile_class, XSD.integer)
create_data_property("NBI_30 - Year of Average Daily Traffic", traffic_profile_class, XSD.gYear)

# ExternalFactor -> EnvironmentalExposure Properties
create_data_property("Avg Daily Temperature", environmental_exposure_class, XSD.decimal)
create_data_property("Total Precipitation", environmental_exposure_class, XSD.decimal)
create_data_property("Number of Snowfall Days", environmental_exposure_class, XSD.integer)

try:
    g.serialize(destination="structured_bridge_ontology.owl", format="xml", encoding="utf-8")
except Exception as e:
    print(f"An error occurred while saving the file: {e}")



Success! Ontology schema file 'structured_bridge_ontology.owl' has been created.
You can now open this file in Protégé to see the new, improved structure.
